# 2.2 Advanced Circuits

This notebook focuses on parameterized layers, reusable subcircuits, and controlled custom gates.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

In [ ]:
def build_parameterized_layer(theta: ParameterVector) -> QuantumCircuit:
    layer = QuantumCircuit(3, name="theta_layer")
    for qubit, angle in enumerate(theta):
        layer.ry(angle, qubit)
    layer.cx(0, 1)
    layer.cx(1, 2)
    layer.rz(theta[0] - theta[2], 2)
    return layer

theta = ParameterVector("theta", 3)
layer = build_parameterized_layer(theta)
layer

In [ ]:
qc = QuantumCircuit(3)
for qubit in range(3):
    qc.h(qubit)
qc.compose(layer, inplace=True)
qc.barrier()
qc.compose(layer.inverse(), inplace=True)

bound = qc.assign_parameters([np.pi / 6, np.pi / 4, np.pi / 3])
bound

In [ ]:
state = Statevector.from_instruction(bound)
print("Depth:", bound.depth())
print("Parameters remaining:", len(bound.parameters))
print("Probabilities:", state.probabilities_dict(decimals=4))

## Reusable Subcircuits

In [ ]:
bell_block = QuantumCircuit(2, name="bell_block")
bell_block.h(0)
bell_block.cx(0, 1)

qc_pairs = QuantumCircuit(4)
qc_pairs.compose(bell_block, qubits=[0, 1], inplace=True)
qc_pairs.compose(bell_block, qubits=[2, 3], inplace=True)
qc_pairs

In [ ]:
pair_state = Statevector.from_instruction(qc_pairs)
print(pair_state.to_dict(decimals=4))

## Controlled Custom Gates

In [ ]:
swap_like = QuantumCircuit(2, name="swap_like")
swap_like.cx(0, 1)
swap_like.cx(1, 0)
swap_like.cx(0, 1)

controlled_gate = swap_like.to_gate().control(1)

controlled_demo = QuantumCircuit(3)
controlled_demo.x(0)
controlled_demo.x(1)
controlled_demo.append(controlled_gate, [0, 1, 2])
controlled_demo

In [ ]:
controlled_state = Statevector.from_instruction(controlled_demo)
print(controlled_state.to_dict())

## Practice

1. Add a second parameterized layer and compare `depth()`.
2. Replace the controlled swap-like gate with a controlled Bell block.
3. Compose `bell_block.inverse()` after `qc_pairs` and verify that the final state is `|0000>`.